# Phase P6: Run C — Reversible Max Batch Training
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thinkTinkerGreen/train_with_reversibility/blob/main/notebooks/05_reversible_max_batch.ipynb)


In [ ]:
import os, sys, json, subprocess
SMOKE = os.environ.get("REVLLM_SMOKE", "0") == "1"
IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/thinkTinkerGreen/train_with_reversibility.git"
REPO_REF = "main"

if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/revllm/data', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/revllm/runs', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/revllm/probes', exist_ok=True)

    repo_dir = "/content/train_with_reversibility"
    if not os.path.exists(repo_dir):
        print(f"Cloning {REPO_URL} into {repo_dir}...")
        subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
    else:
        print(f"Repository already present at {repo_dir}. Fetching updates...")
        subprocess.run(["git", "-C", repo_dir, "fetch", "--all"], check=False)
        subprocess.run(["git", "-C", repo_dir, "reset", "--hard", f"origin/{REPO_REF}"], check=False)

    os.chdir(repo_dir)

    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    src_dir = os.path.join(repo_dir, "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

    # Ensure dependencies and package are installed in fresh Colab runtimes
    deps = ["hatchling", "pyyaml", "tokenizers", "huggingface-hub", "datasets", "tqdm"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + deps, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
print(f"Current working directory: {os.getcwd()}")
print(f"Execution mode: {'Smoke (CPU/CI)' if SMOKE else 'Colab GPU'}")


In [ ]:
from revllm.config import ModelConfig, RunConfig, TrainConfig
from revllm.train import train_run

# Restore dataset from Drive if needed
if IN_COLAB and not SMOKE:
    os.makedirs("data", exist_ok=True)
    for fn in ["train.bin", "val.bin", "tokenizer.json"]:
        drive_fp = f"/content/drive/MyDrive/revllm/data/{fn}"
        if os.path.exists(drive_fp) and not os.path.exists(f"data/{fn}"):
            import shutil
            shutil.copy2(drive_fp, f"data/{fn}")

model_cfg = ModelConfig.from_yaml("configs/model.yaml")
train_cfg = TrainConfig.from_yaml("configs/train.yaml")
run_cfg = RunConfig.from_yaml("configs/runs/C.yaml")

# Load winner variant from pilot
v_path = "results/variant_selection.json"
if not os.path.exists(v_path) and IN_COLAB and not SMOKE:
    drive_v = "/content/drive/MyDrive/revllm/runs/variant_selection.json"
    if os.path.exists(drive_v):
        import shutil
        os.makedirs("results", exist_ok=True)
        shutil.copy2(drive_v, v_path)

if os.path.exists(v_path):
    with open(v_path) as f:
        v_data = json.load(f)
        run_cfg.arch = v_data.get("winner", run_cfg.arch)
        run_cfg.h = 0.5 if run_cfg.arch == "midpoint" else 1.0

# Batch size MUST be max safe batch for the reversible variant (B_C)
probe_fp = "results/batch_probe.json"
if not os.path.exists(probe_fp) and IN_COLAB and not SMOKE:
    drive_probe = "/content/drive/MyDrive/revllm/probes/batch_probe.json"
    if os.path.exists(drive_probe):
        import shutil
        os.makedirs("results", exist_ok=True)
        shutil.copy2(drive_probe, probe_fp)

if os.path.exists(probe_fp):
    with open(probe_fp) as f:
        probes = json.load(f)
        run_cfg.batch_size = probes.get("b_C", run_cfg.batch_size)

drive_dir = "/content/drive/MyDrive/revllm/runs/rev_max" if IN_COLAB and not SMOKE else None
result = train_run(
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    run_cfg=run_cfg,
    train_data_path="data/train.bin",
    val_data_path="data/val.bin",
    local_output_dir="results/runs/C",
    drive_output_dir=drive_dir,
    smoke_mode=SMOKE,
)
print("Run C Completed! Final Result:", json.dumps(result, indent=2))
